# 02 — Data Cleaning

Documented, auditable fix for each issue found in `01_eda.ipynb`. Same rule as the retail project: never silently delete data that might be real — flag it, count it, and let downstream queries decide whether to include it.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path('data/raw')
employees = pd.read_csv(RAW / 'employees.csv')
compensation = pd.read_csv(RAW / 'compensation.csv')
performance = pd.read_csv(RAW / 'performance_reviews.csv')
exit_surveys = pd.read_csv(RAW / 'exit_surveys.csv')
departments = pd.read_csv(RAW / 'departments.csv')
print(f'Starting employee rows: {len(employees):,}')

Starting employee rows: 3,347


### Fix 1 — drop duplicate employee_id rows

In [2]:
before = len(employees)
employees = employees.drop_duplicates(subset=['employee_id'], keep='first')
print(f'Removed {before - len(employees)} duplicates. Remaining: {len(employees):,}')

Removed 47 duplicates. Remaining: 3,300


### Fix 2 — standardize department labels

In [3]:
CANONICAL = {d.strip().upper(): d for d in departments['department_name']}
def clean_dept(d):
    return CANONICAL.get(str(d).strip().upper(), str(d).strip())
before_n = employees['department'].nunique()
employees['department'] = employees['department'].apply(clean_dept)
print(f"Department labels: {before_n} raw variants -> {employees['department'].nunique()} canonical values")

Department labels: 27 raw variants -> 7 canonical values


### Fix 3 — flag the status-sync gap instead of guessing a termination date

In [4]:
employees['hire_date'] = pd.to_datetime(employees['hire_date'], errors='coerce')
employees['termination_date'] = pd.to_datetime(employees['termination_date'], errors='coerce')
gap = (employees['active_status']==0) & (employees['termination_date'].isna())
employees['data_quality_flag'] = None
employees.loc[gap, 'data_quality_flag'] = 'MISSING_TERM_DATE'
print(f'{gap.sum()} rows flagged MISSING_TERM_DATE (not imputed — a guessed exit date would corrupt tenure analysis).')

122 rows flagged MISSING_TERM_DATE (not imputed — a guessed exit date would corrupt tenure analysis).


### Fix 4 — correct swapped hire/termination dates

In [5]:
swapped = employees['termination_date'].notna() & (employees['hire_date'] > employees['termination_date'])
print(f'{swapped.sum()} rows had swapped dates — correcting.')
tmp = employees.loc[swapped, 'hire_date'].copy()
employees.loc[swapped, 'hire_date'] = employees.loc[swapped, 'termination_date']
employees.loc[swapped, 'termination_date'] = tmp

33 rows had swapped dates — correcting.


### Fix 5 & 6 — flag invalid distance and invalid salary (don't impute)

In [6]:
employees['is_valid_distance'] = employees['distance_from_home_miles'] > 0
compensation['is_valid_salary'] = compensation['base_salary'] > 0
print(f"{(~employees['is_valid_distance']).sum()} invalid distance rows flagged.")
print(f"{(~compensation['is_valid_salary']).sum()} invalid salary rows flagged.")

58 invalid distance rows flagged.
61 invalid salary rows flagged.


### Fix 7 — flag invalid performance scores

In [7]:
performance['is_valid_score'] = performance['performance_score'].between(1,5)
print(f"{(~performance['is_valid_score']).sum()} invalid performance scores flagged.")

41 invalid performance scores flagged.


### Fix 8 — drop orphaned exit-survey rows

In [8]:
before_exit = len(exit_surveys)
exit_surveys = exit_surveys[exit_surveys['employee_id'].isin(employees['employee_id'])]
print(f'Dropped {before_exit - len(exit_surveys)} orphaned exit survey rows.')

Dropped 24 orphaned exit survey rows.


### Save cleaned tables

In [9]:
Path('data/processed').mkdir(exist_ok=True)
employees.to_csv('data/processed/employees_clean.csv', index=False)
compensation.to_csv('data/processed/compensation_clean.csv', index=False)
performance.to_csv('data/processed/performance_reviews_clean.csv', index=False)
exit_surveys.to_csv('data/processed/exit_surveys_clean.csv', index=False)
departments.to_csv('data/processed/departments_clean.csv', index=False)
print(f"Final employees table: {len(employees):,} rows")

Final employees table: 3,300 rows
